In [ ]:
# =========================================
# FINAL TESLA STOCK PREDICTION MODEL
# =========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, LSTM
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Input   # 🔥 IMPORTANT FIX

print("Starting Project...")

# =========================================
# LOAD DATA
# =========================================
df = pd.read_csv('../data/TSLA.csv')

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.set_index('Date', inplace=True)

df = df.ffill()   # ✅ FIX for pandas

# =========================================
# VISUALIZATION
# =========================================
plt.figure(figsize=(12,5))
plt.plot(df['Adj Close'], label='Adj Close')
plt.title("Tesla Adjusted Close Price")
plt.legend()
plt.show()

# =========================================
# FEATURE SELECTION
# =========================================
data = df[['Adj Close']]

# =========================================
# SCALING
# =========================================
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

# =========================================
# CREATE DATASET
# =========================================
def create_dataset(data, time_step=60):
    X, y = [], []
    for i in range(len(data)-time_step-1):
        X.append(data[i:(i+time_step), 0])
        y.append(data[i+time_step, 0])
    return np.array(X), np.array(y)

time_step = 60
X, y = create_dataset(scaled_data, time_step)

X = X.reshape(X.shape[0], X.shape[1], 1)

# =========================================
# TRAIN TEST SPLIT
# =========================================
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# =========================================
# RNN MODEL (FIXED)
# =========================================
model_rnn = Sequential([
    Input(shape=(60,1)),   # 🔥 FIX
    SimpleRNN(50, return_sequences=True),
    Dropout(0.2),
    SimpleRNN(50),
    Dropout(0.2),
    Dense(1)
])

model_rnn.compile(optimizer='adam', loss='mean_squared_error')

early_stop = EarlyStopping(patience=5)

print("Training RNN...")
model_rnn.fit(X_train, y_train,
              epochs=20,
              batch_size=32,
              validation_data=(X_test, y_test),
              callbacks=[early_stop])

# =========================================
# LSTM MODEL (FINAL FIXED)
# =========================================
model_lstm = Sequential([
    Input(shape=(60,1)),   # 🔥 MOST IMPORTANT FIX
    LSTM(50, return_sequences=True),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')

print("Training LSTM...")

history = model_lstm.fit(X_train, y_train,
                         epochs=20,
                         batch_size=32,
                         validation_data=(X_test, y_test),
                         callbacks=[early_stop])

# =========================================
# PREDICTIONS
# =========================================
pred_lstm = model_lstm.predict(X_test)
pred_lstm = scaler.inverse_transform(pred_lstm)

y_test_actual = scaler.inverse_transform(y_test.reshape(-1,1))

# =========================================
# PLOT RESULTS
# =========================================
plt.figure(figsize=(12,6))
plt.plot(y_test_actual, label='Actual')
plt.plot(pred_lstm, label='LSTM')
plt.legend()
plt.title("Actual vs Predicted")
plt.show()

# =========================================
# EVALUATION
# =========================================
mse = mean_squared_error(y_test_actual, pred_lstm)
print("LSTM MSE:", mse)

# =========================================
# FUTURE PREDICTION
# =========================================
def predict_future(days):
    temp_input = list(scaled_data[-60:])
    predictions = []

    for i in range(days):
        x_input = np.array(temp_input[-60:])
        x_input = x_input.reshape(1,60,1)

        pred = model_lstm.predict(x_input, verbose=0)
        temp_input.append(pred[0])
        predictions.append(pred[0][0])

    return scaler.inverse_transform(np.array(predictions).reshape(-1,1))

print("\nFuture Predictions:")
for d in [1,5,10]:
    print(f"{d} days:", predict_future(d).flatten())

# =========================================
# SAVE MODEL (FINAL)
# =========================================
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
model_dir = os.path.join(BASE_DIR, "outputs", "models")

os.makedirs(model_dir, exist_ok=True)

model_lstm.save(os.path.join(model_dir, "lstm_model.keras"))   # ✅ FINAL FORMAT

print("\nModel saved successfully!")

In [ ]:
## Insights

- LSTM performs better than RNN due to long-term dependency handling  
- Stock prediction is influenced by market volatility  
- Model captures trends but cannot predict sudden changes  

## Hyperparameter Tuning

Different configurations of units and dropout were tested manually due to limitations of GridSearchCV with deep learning models.